In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import circmean, circstd, ttest_ind, mannwhitneyu
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings('ignore')
nature_colors = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F']
science_colors = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#F0E442']
import numpy as np
from scipy.stats import ttest_ind, mannwhitneyu, shapiro

def two_group_comparison(group1, group2, alpha=0.05):
    """
    Perform two-group comparison with automatic test selection based on normality.

    Parameters
    ----------
    group1, group2 : array-like
        Input data for the two groups.
    alpha : float
        Significance level for normality test.

    Returns
    -------
    result : dict
        Dictionary containing test name, statistic, and p-value.
    """

    # Convert to numpy arrays and remove NaN values
    g1 = np.asarray(group1)
    g2 = np.asarray(group2)
    g1 = g1[~np.isnan(g1)]
    g2 = g2[~np.isnan(g2)]

    # Initialize normality p-values
    p_norm1, p_norm2 = None, None

    # Perform Shapiro-Wilk test for normality (if sample size is sufficient)
    if len(g1) >= 3 and len(g2) >= 3:
        _, p_norm1 = shapiro(g1)
        _, p_norm2 = shapiro(g2)

        # Data are considered normal only if both groups pass the test
        is_normal = (p_norm1 > alpha) and (p_norm2 > alpha)
    else:
        # For very small sample sizes, default to non-parametric test
        is_normal = False

    # Select appropriate statistical test
    if is_normal:
        # Independent samples t-test (parametric)
        stat, p_value = ttest_ind(g1, g2, nan_policy='omit')
        test_name = 'independent t-test'
    else:
        # Mann-Whitney U test (non-parametric)
        stat, p_value = mannwhitneyu(g1, g2, alternative='two-sided')
        test_name = 'Mann-Whitney U test'

    return {
        'test': test_name,
        'statistic': stat,
        'p_value': p_value,
        'normality_p_group1': p_norm1,
        'normality_p_group2': p_norm2
    }

# Custom detrend function
def detrend(signal):
    """Simple linear detrending"""
    x = np.arange(len(signal))
    coeffs = np.polyfit(x, signal, 1)
    trend = np.polyval(coeffs, x)
    return signal - trend

class ThetaGammaPAC:
    """
    Class for analyzing theta-gamma phase-amplitude coupling
    """
    
    def __init__(self, fs=1000):
        self.fs = fs
        
    def filter_data(self, data, lowcut, highcut, order=4):
        """Bandpass filter the data"""
        nyquist = 0.5 * self.fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = butter(order, [low, high], btype='band')
        return filtfilt(b, a, data)
    
    def get_phase_and_amplitude(self, data, theta_range=(4, 8), gamma_range=(30, 80)):
        """Extract theta phase and gamma amplitude"""
        theta_filtered = self.filter_data(data, theta_range[0], theta_range[1])
        gamma_filtered = self.filter_data(data, gamma_range[0], gamma_range[1])
        
        theta_analytic = hilbert(theta_filtered)
        gamma_analytic = hilbert(gamma_filtered)
        
        theta_phase = np.angle(theta_analytic)
        gamma_amp = np.abs(gamma_analytic)
        
        return theta_phase, gamma_amp, theta_filtered, gamma_filtered
    
    def compute_mi(self, data, theta_range=(4, 8), gamma_range=(30, 80), n_bins=18):
        """Compute Modulation Index (MI)"""
        theta_phase, gamma_amp, _, _ = self.get_phase_and_amplitude(
            data, theta_range, gamma_range
        )
        
        phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
        bin_indices = np.digitize(theta_phase, phase_bins) - 1
        
        amp_by_phase = np.zeros(n_bins)
        for i in range(n_bins):
            mask = bin_indices == i
            if np.any(mask):
                amp_by_phase[i] = np.mean(gamma_amp[mask])
        
        p = amp_by_phase / (np.sum(amp_by_phase) + 1e-12)
        H = -np.sum(p * np.log(p + 1e-12))
        H_uniform = np.log(n_bins)
        KL = H_uniform - H
        MI = KL / H_uniform
        
        return MI, phase_bins, amp_by_phase, theta_phase, gamma_amp
    
    def compute_mvl(self, data, theta_range=(4, 8), gamma_range=(30, 80)):
        """Compute Mean Vector Length (MVL)"""
        theta_phase, gamma_amp, _, _ = self.get_phase_and_amplitude(
            data, theta_range, gamma_range
        )
        
        complex_vector = gamma_amp * np.exp(1j * theta_phase)
        mean_vector = np.mean(complex_vector)
        MVL = np.abs(mean_vector)
        preferred_phase = np.angle(mean_vector)
        
        return MVL, preferred_phase, theta_phase, gamma_amp


# ============================================================================
# Main comparison analysis
# ============================================================================

# Initialize PAC analyzer
pac = ThetaGammaPAC(fs=1000)

# Store results for both conditions
bio_results = []
hard_results = []

# Loop through all 10 files for both conditions
for i in range(10):
    print(f"\n{'='*50}")
    print(f"Processing pair {i+1}/10")
    print('='*50)
    
    # File paths
    filedir = "../Simulations/rand3/"
    filepath_bio = f"{filedir}rand_bio{i}/out/"
    filepath_hard = f"{filedir}rand_hard{i}/out/"
    
    
    # Process BIO data
    try:
        print(f"\nProcessing BIO data for file {i}...")
        
        # Load BIO data
        time_cx_bio = np.loadtxt(filepath_bio + "time_cx")
        #index = np.arange(10000, time_cx_bio.shape[0])
        index = np.arange(40200, 40200+5000)
        time_bio = (index - index[0]) / 1000
        data_bio = time_cx_bio[index, 1:-1].T
        
        # Load external current
        bio_current = np.loadtxt(filepath_bio + "bio_current.dat")
        Iext_bio = bio_current[index, 1]
        
        # Compute LFP
        LFP_bio = np.sum(data_bio, axis=0) / data_bio.shape[0]
        LFP_bio = detrend(LFP_bio)
        
        # Compute PAC metrics
        MI_bio, _, amp_by_phase_bio, theta_phase_bio, gamma_amp_bio = pac.compute_mi(LFP_bio)
        MVL_bio, pref_phase_bio, _, _ = pac.compute_mvl(LFP_bio)
        
        # Store results
        bio_results.append({
            'file_index': i,
            'MI': MI_bio,
            'MVL': MVL_bio,
            'preferred_phase': pref_phase_bio,
            'preferred_phase_deg': pref_phase_bio * 180 / np.pi,
            'LFP': LFP_bio,
            'time': time_bio,
            'Iext': Iext_bio,
            'amp_by_phase': amp_by_phase_bio,
            'theta_phase': theta_phase_bio,
            'gamma_amp': gamma_amp_bio
        })
        
        print(f"  BIO - MI: {MI_bio:.4f}, MVL: {MVL_bio:.4f}, Phase: {pref_phase_bio*180/np.pi:.1f}°")
        
    except Exception as e:
        print(f"Error processing BIO file {i}: {e}")
        bio_results.append({'file_index': i, 'MI': np.nan, 'MVL': np.nan, 
                           'preferred_phase': np.nan, 'error': str(e)})
    
    # Process HARD data
    try:
        print(f"\nProcessing HARD data for file {i}...")
        
        # Load HARD data
        time_cx_hard = np.loadtxt(filepath_hard + "time_cx")
        data_hard = time_cx_hard[index, 1:-1].T
        
        # Load external current
        hard_current = np.loadtxt(filepath_hard + "bio_current.dat")
        Iext_hard = hard_current[index, 1]
        
        # Compute LFP
        LFP_hard = np.sum(data_hard, axis=0) / data_hard.shape[0]
        LFP_hard = detrend(LFP_hard)
        
        # Compute PAC metrics
        MI_hard, _, amp_by_phase_hard, theta_phase_hard, gamma_amp_hard = pac.compute_mi(LFP_hard)
        MVL_hard, pref_phase_hard, _, _ = pac.compute_mvl(LFP_hard)
        
        # Store results
        hard_results.append({
            'file_index': i,
            'MI': MI_hard,
            'MVL': MVL_hard,
            'preferred_phase': pref_phase_hard,
            'preferred_phase_deg': pref_phase_hard * 180 / np.pi,
            'LFP': LFP_hard,
            'time': time_bio,  # Same time base
            'Iext': Iext_hard,
            'amp_by_phase': amp_by_phase_hard,
            'theta_phase': theta_phase_hard,
            'gamma_amp': gamma_amp_hard
        })
        
        print(f"  HARD - MI: {MI_hard:.4f}, MVL: {MVL_hard:.4f}, Phase: {pref_phase_hard*180/np.pi:.1f}°")
        
    except Exception as e:
        print(f"Error processing HARD file {i}: {e}")
        hard_results.append({'file_index': i, 'MI': np.nan, 'MVL': np.nan, 
                            'preferred_phase': np.nan, 'error': str(e)})

# ============================================================================
# Extract metrics for comparison (filter out NaN values)
# ============================================================================

bio_MI = np.array([r['MI'] for r in bio_results if not np.isnan(r['MI'])])
hard_MI = np.array([r['MI'] for r in hard_results if not np.isnan(r['MI'])])

bio_MVL = np.array([r['MVL'] for r in bio_results if not np.isnan(r['MVL'])])
hard_MVL = np.array([r['MVL'] for r in hard_results if not np.isnan(r['MVL'])])

bio_phase = np.array([r['preferred_phase'] for r in bio_results if not np.isnan(r['preferred_phase'])])
hard_phase = np.array([r['preferred_phase'] for r in hard_results if not np.isnan(r['preferred_phase'])])

bio_phase_deg = np.array([r['preferred_phase_deg'] for r in bio_results if not np.isnan(r['preferred_phase_deg'])])
hard_phase_deg = np.array([r['preferred_phase_deg'] for r in hard_results if not np.isnan(r['preferred_phase_deg'])])


# MI comparison
result_mi = two_group_comparison(bio_MI, hard_MI)
t_stat_mi = result_mi['statistic']
p_value_mi = result_mi['p_value']

result_mvl = two_group_comparison(bio_MVL, hard_MVL)
t_stat_mvl = result_mvl['statistic']
p_value_mvl = result_mvl['p_value']


# Calculate average phase-amplitude histograms
n_bins = 18
phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
bin_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

# Collect all amp_by_phase arrays
bio_amp_phases = np.array([r['amp_by_phase'] for r in bio_results if 'amp_by_phase' in r])
hard_amp_phases = np.array([r['amp_by_phase'] for r in hard_results if 'amp_by_phase' in r])

# ============================================================================
# Save results
# ============================================================================
# Save all results to file
np.savez(filedir+'theta_gamma_bio_vs_hard_results_new.npz',
         bio_MI=bio_MI,
         hard_MI=hard_MI,
         bio_MVL=bio_MVL,
         hard_MVL=hard_MVL,
         bio_amp_phases=bio_amp_phases,
         hard_amp_phases=hard_amp_phases,
         p_value_mvl=p_value_mvl,
         p_value_mi=p_value_mi
         )

print("\n" + "="*60)
print("RESULTS SAVED: theta_gamma_bio_vs_hard_results.npz")
print("FIGURES SAVED:")
print("  - theta_gamma_bio_vs_hard_comparison.png")
print("  - theta_gamma_phase_amplitude_histograms.png")
print("  - theta_gamma_paired_differences.png")
print("="*60)
print("\n✓ Comparison analysis complete!")


Processing pair 1/10

Processing BIO data for file 0...
  BIO - MI: 0.0030, MVL: 0.0089, Phase: 7.2°

Processing HARD data for file 0...
  HARD - MI: 0.0061, MVL: 0.0335, Phase: 7.3°

Processing pair 2/10

Processing BIO data for file 1...
  BIO - MI: 0.0032, MVL: 0.0185, Phase: -19.3°

Processing HARD data for file 1...
  HARD - MI: 0.0078, MVL: 0.0426, Phase: -27.9°

Processing pair 3/10

Processing BIO data for file 2...
  BIO - MI: 0.0058, MVL: 0.0242, Phase: -20.1°

Processing HARD data for file 2...
  HARD - MI: 0.0042, MVL: 0.0238, Phase: -13.2°

Processing pair 4/10

Processing BIO data for file 3...
  BIO - MI: 0.0053, MVL: 0.0385, Phase: -21.9°

Processing HARD data for file 3...
  HARD - MI: 0.0042, MVL: 0.0293, Phase: -1.6°

Processing pair 5/10

Processing BIO data for file 4...
  BIO - MI: 0.0031, MVL: 0.0217, Phase: -39.5°

Processing HARD data for file 4...
  HARD - MI: 0.0085, MVL: 0.0320, Phase: -19.5°

Processing pair 6/10

Processing BIO data for file 5...
  BIO - 